In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [10]:
df = pd.read_csv("/Users/karim/Documents/la-rent-anomaly-detection/la-rent-anomaly-detection/data/raw/la_neighborhood_rental_prices.csv", index_col='Neighborhood')

In [11]:
df_filled = df.bfill(axis=1)
print(f"Remaining NaNs: {df_filled.isna().sum().sum()}")

Remaining NaNs: 0


In [12]:
rent_long = df_filled.reset_index().melt(
    id_vars='Neighborhood',
    var_name='Date',
    value_name='Rent'
)
rent_long['Date'] = pd.to_datetime(rent_long['Date'])
rent_long = rent_long.sort_values(['Neighborhood', 'Date']).reset_index(drop=True)
print(rent_long.shape)
print(rent_long.head(10))

(1870, 3)
    Neighborhood       Date         Rent
0  Beverly Hills 2019-01-31  8010.221774
1  Beverly Hills 2019-02-28  7828.068572
2  Beverly Hills 2019-03-31  5275.703166
3  Beverly Hills 2019-04-30  5309.672058
4  Beverly Hills 2019-05-31  5405.183404
5  Beverly Hills 2019-06-30  5524.377234
6  Beverly Hills 2019-07-31  5495.045186
7  Beverly Hills 2019-08-31  5365.272714
8  Beverly Hills 2019-09-30  5289.047364
9  Beverly Hills 2019-10-31  5365.102093


In [14]:
rent_long.to_csv('/Users/karim/Documents/la-rent-anomaly-detection/la-rent-anomaly-detection/data/processed/zillow_rent_clean.csv', index=False)
print("Saved successfully")

Saved successfully


In [2]:
import pandas as pd
df = pd.read_csv('/Users/karim/Documents/la-rent-anomaly-detection/la-rent-anomaly-detection/data/processed/zillow_rent_clean.csv')
df.columns

Index(['Neighborhood', 'Date', 'Rent'], dtype='object')

In [9]:
df['Neighborhood'].unique()

array(['Beverly Hills', 'Boyle Heights', 'Brentwood', 'Culver City',
       'Downtown LA', 'Exposition Park', 'Glassell Park', 'Hancock Park',
       'Highland Park', 'Koreatown', 'Leimert Park', 'Los Feliz',
       'Mar Vista', 'Mid-Wilshire', 'Palms', 'Santa Monica',
       'Silver Lake', 'University Park', 'Vermont Square', 'West Adams',
       'West Hollywood', 'Westwood'], dtype=object)

In [ ]:
import pandas as pd
import numpy as np

# ── Load all sources ──────────────────────────────────────────────────────────
base_path = '/Users/karim/Documents/la-rent-anomaly-detection/la-rent-anomaly-detection/data/'

rent         = pd.read_csv(base_path + 'processed/zillow_rent_clean.csv')
permits      = pd.read_csv(base_path + 'processed/permits_clean.csv')
employment   = pd.read_csv(base_path + 'processed/employment_clean.csv')
demographics = pd.read_csv(base_path + 'processed/demographics_clean.csv')

# ── Standardize Date columns ──────────────────────────────────────────────────
rent['Date']       = pd.to_datetime(rent['Date'])
permits['Date']    = pd.to_datetime(permits['Date'])
employment['Date'] = pd.to_datetime(employment['Date'])

# ── Trim rent to 2019-01 through 2025-12 ─────────────────────────────────────
rent = rent[rent['Date'] <= '2025-12-31'].copy()
print(f"Rent after trim: {rent.shape}")

# ── Merge 1: rent + permits ───────────────────────────────────────────────────
master = rent.merge(permits, on=['Neighborhood', 'Date'], how='left')
print(f"After permits merge: {master.shape}")

# Fill NaN permit values for pre-2020 rows with 0
permit_cols = [c for c in permits.columns if c not in ['Neighborhood', 'Date']]
master[permit_cols] = master[permit_cols].fillna(0)
print(f"Permit NaNs after fill: {master[permit_cols].isnull().sum().sum()}")

# ── Merge 2: + employment ─────────────────────────────────────────────────────
master = master.merge(employment[['Date', 'employment_level', 'employment_growth']],
                      on='Date', how='left')
print(f"After employment merge: {master.shape}")

# ── Merge 3: + demographics ───────────────────────────────────────────────────
# Extract year from Date for join key
master['year'] = master['Date'].dt.year

# Forward fill 2024/2025 using 2023 values
demo_2023 = demographics[demographics['year'] == 2023].copy()

for yr in [2024, 2025]:
    demo_yr = demo_2023.copy()
    demo_yr['year'] = yr
    demographics = pd.concat([demographics, demo_yr], ignore_index=True)

master = master.merge(demographics, on=['Neighborhood', 'year'], how='left')
print(f"After demographics merge: {master.shape}")

# ── Feature Engineering ───────────────────────────────────────────────────────

# 1. COVID indicator
master['covid'] = (
    (master['Date'] >= '2020-03-01') &
    (master['Date'] <= '2021-06-30')
).astype(int)

# 2. Seasonality
master['month']   = master['Date'].dt.month
master['quarter'] = master['Date'].dt.quarter

# 3. Rent lag features — always groupby Neighborhood first
master = master.sort_values(['Neighborhood', 'Date'])

master['rent_lag1']  = master.groupby('Neighborhood')['Rent'].shift(1)
master['rent_lag3']  = master.groupby('Neighborhood')['Rent'].shift(3)
master['rent_lag6']  = master.groupby('Neighborhood')['Rent'].shift(6)
master['rent_lag12'] = master.groupby('Neighborhood')['Rent'].shift(12)

# 4. Rent rolling features
master['rent_rolling3_mean'] = (
    master.groupby('Neighborhood')['Rent']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)
master['rent_rolling6_mean'] = (
    master.groupby('Neighborhood')['Rent']
    .transform(lambda x: x.rolling(6, min_periods=1).mean())
)

# 5. Rent MoM and YoY growth
master['rent_mom_growth'] = (
    master.groupby('Neighborhood')['Rent']
    .transform(lambda x: x.pct_change(1) * 100)
)
master['rent_yoy_growth'] = (
    master.groupby('Neighborhood')['Rent']
    .transform(lambda x: x.pct_change(12) * 100)
)

# ── Final cleanup ─────────────────────────────────────────────────────────────
# Drop helper year column
master = master.drop(columns=['year'])

# Sort
master = master.sort_values(['Neighborhood', 'Date']).reset_index(drop=True)

keep_cols = [
    'Neighborhood', 'Date', 'Rent',
    'permits_trailing6',
    'employment_growth',
    'median_income', 'renter_rate',
    'covid', 'month',
    'rent_lag1', 'rent_lag12',
    'rent_yoy_growth'
]

master = master[keep_cols]

# ── Sanity checks ─────────────────────────────────────────────────────────────
print(f"\n── MASTER TABLE ──")
print(f"Shape: {master.shape}")
print(f"Neighborhoods: {master['Neighborhood'].nunique()}")
print(f"Date range: {master['Date'].min()} to {master['Date'].max()}")
print(f"Months per neighborhood: {master.groupby('Neighborhood')['Date'].count().unique()}")
print(f"\nColumns:\n{list(master.columns)}")
print(f"\nMissing values:\n{master.isnull().sum()}")
print(f"\nSample:\n{master.head()}")

# ── Save ──────────────────────────────────────────────────────────────────────
output_path = base_path + 'processed/master.csv'
master.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")

In [ ]:
master['rent_lag1'] = master.groupby('Neighborhood')['rent_lag1'].transform(
    lambda x: x.fillna(x.iloc[1] if len(x) > 1 else x)
)

master['employment_growth'] = master['employment_growth'].fillna(0)